# Day 4 — Winter ONI forecast + city impacts

1. Load latest 12 Pacific SST anomaly maps + a trained CNN  
2. Forecast ONI for winter **2026–27**  
3. Pull city winter histories from Open-Meteo  
4. Fit ONI → winter anomaly per city → write `impacts.json`

**Watch:** December belongs to the *following* winter (Dec 2026 → winter 2027). Anomalies are vs each city's own long-term winter mean.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarib-sami/ninonet/blob/main/enso/day4_impacts.ipynb)

Prefer running **Day 3 tuned** first so `enso_cnn_lead6_tuned.pt` (or lead3) exists. Falls back to untuned checkpoints.


## 0. Install


In [ ]:
!pip install -q xarray netCDF4 numpy pandas scikit-learn matplotlib requests torch


## 1. Mount Drive


In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/ensocast/data")
OUT_DIR = Path("/content/drive/MyDrive/ensocast/artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / "pacific_anom.nc").exists(), "Rerun Day 1"
assert (DATA_DIR / "oni_monthly.csv").exists(), "Rerun Day 1"
print("Data:", DATA_DIR)
print("Artifacts:", OUT_DIR)


## 2. Load SST anomalies + ONI


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

anom = xr.open_dataarray(DATA_DIR / "pacific_anom.nc")
if isinstance(anom, xr.Dataset):
    anom = anom[list(anom.data_vars)[0]]

oni_df = pd.read_csv(DATA_DIR / "oni_monthly.csv", parse_dates=["time"])

arr = anom.values.astype("float32")
times = pd.to_datetime(anom["time"].values).to_period("M").to_timestamp()

oni_series = oni_df.set_index("time")["oni"]
oni_series.index = pd.to_datetime(oni_series.index).to_period("M").to_timestamp()
oni = oni_series.reindex(times).to_numpy(dtype="float32")

missing = int(np.isnan(oni).sum())
if missing:
    print(f"Dropping {missing} month(s) with no ONI.")
    valid = ~np.isnan(oni)
    arr, times, oni = arr[valid], times[valid], oni[valid]

print("months:", len(arr), "last:", times[-1].date(), "last ONI:", float(oni[-1]))


## 3. Load CNN and forecast winter ONI

Default: **lead 6** (beat persistence in Day 3). Override with `LEAD = 3` if you prefer.

Winter 2026–27 ONI ≈ DJF centered on **Jan 2027**.


In [ ]:
import torch
import torch.nn as nn

LEAD = 6  # or 3
WINDOW = 12

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class ENSOForecaster(nn.Module):
    # Matches Day 3 tuned (BatchNorm). Also loads older ckpts if shapes match.
    def __init__(self, in_months=12, dropout=0.4, use_bn=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_months, 32, 3, padding=1),
        ]
        if use_bn:
            layers.append(nn.BatchNorm2d(32))
        layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1)]
        if use_bn:
            layers.append(nn.BatchNorm2d(64))
        layers += [
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


candidates = [
    OUT_DIR / f"enso_cnn_lead{LEAD}_tuned.pt",
    OUT_DIR / f"enso_cnn_lead{LEAD}.pt",
    OUT_DIR / "enso_cnn_lead3_tuned.pt",
    OUT_DIR / "enso_cnn_lead3.pt",
]
ckpt_path = next((p for p in candidates if p.exists()), None)
assert ckpt_path is not None, "No CNN checkpoint — rerun Day 2/3 (prefer tuned Day 3)"
print("Loading", ckpt_path)

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt["model_state"]
use_bn = any("running_mean" in k for k in state)
dropout = float(ckpt.get("hp", {}).get("dropout", 0.4)) if isinstance(ckpt.get("hp"), dict) else 0.4

model = ENSOForecaster(dropout=dropout, use_bn=use_bn).to(device)
model.load_state_dict(state)
model.eval()
print("use_bn:", use_bn, "lead in ckpt:", ckpt.get("lead"), "test_rmse:", ckpt.get("test_rmse"))


In [ ]:
# Latest WINDOW months as input
x = arr[-WINDOW:]  # (12, lat, lon)
mu = ckpt.get("norm_mu")
sd = ckpt.get("norm_sd")
if mu is not None and sd is not None:
    x = (x - float(mu)) / float(sd)
    print(f"applied train norm mu={mu:.4f} sd={sd:.4f}")
else:
    print("no norm in checkpoint (untuned Day 2/3)")

x_t = torch.from_numpy(x.astype("float32")[None]).to(device)  # (1, 12, lat, lon)
with torch.no_grad():
    forecast_oni = float(model(x_t).cpu().numpy().squeeze())

last_month = times[-1]
# Rough target month = last input + lead
target_month = (last_month + pd.offsets.MonthBegin(ckpt.get("lead", LEAD)))
print(f"input window ends: {last_month.date()}")
print(f"model lead: {ckpt.get('lead', LEAD)} -> ~target {target_month.date()}")
print(f"forecast ONI: {forecast_oni:.3f}")

# Winter label for the demo
WINTER_LABEL = "2026-27"
print(f"Using this as winter {WINTER_LABEL} ONI forecast for impacts.")


## 4. City list (North America)

~55 recognizable cities. Cut to ~30 later if Open-Meteo is slow.


In [ ]:
CITIES = [
    ("Vancouver", 49.28, -123.12),
    ("Seattle", 47.61, -122.33),
    ("Portland", 45.52, -122.68),
    ("San Francisco", 37.77, -122.42),
    ("Los Angeles", 34.05, -118.24),
    ("San Diego", 32.72, -117.16),
    ("Sacramento", 38.58, -121.49),
    ("Phoenix", 33.45, -112.07),
    ("Las Vegas", 36.17, -115.14),
    ("Salt Lake City", 40.76, -111.89),
    ("Denver", 39.74, -104.99),
    ("Albuquerque", 35.08, -106.65),
    ("Calgary", 51.05, -114.07),
    ("Edmonton", 53.55, -113.49),
    ("Winnipeg", 49.90, -97.14),
    ("Minneapolis", 44.98, -93.27),
    ("Chicago", 41.88, -87.63),
    ("Detroit", 42.33, -83.05),
    ("Toronto", 43.65, -79.38),
    ("Ottawa", 45.42, -75.70),
    ("Montreal", 45.50, -73.57),
    ("Quebec City", 46.81, -71.21),
    ("Boston", 42.36, -71.06),
    ("New York", 40.71, -74.01),
    ("Philadelphia", 39.95, -75.17),
    ("Washington DC", 38.91, -77.04),
    ("Pittsburgh", 40.44, -79.99),
    ("Columbus", 39.96, -83.00),
    ("Indianapolis", 39.77, -86.16),
    ("Nashville", 36.16, -86.78),
    ("Atlanta", 33.75, -84.39),
    ("Charlotte", 35.23, -80.84),
    ("Miami", 25.76, -80.19),
    ("Tampa", 27.95, -82.46),
    ("New Orleans", 29.95, -90.07),
    ("Houston", 29.76, -95.37),
    ("Dallas", 32.78, -96.80),
    ("Austin", 30.27, -97.74),
    ("San Antonio", 29.42, -98.49),
    ("Oklahoma City", 35.47, -97.52),
    ("Kansas City", 39.10, -94.58),
    ("St. Louis", 38.63, -90.20),
    ("Omaha", 41.26, -95.94),
    ("Kansas City KS", 39.11, -94.63),
    ("Anchorage", 61.22, -149.90),
    ("Honolulu", 21.31, -157.86),
    ("Mexico City", 19.43, -99.13),
    ("Monterrey", 25.67, -100.31),
    ("Guadalajara", 20.67, -103.35),
    ("Tijuana", 32.51, -117.04),
    ("Halifax", 44.65, -63.57),
    ("St. John's", 47.56, -52.71),
    ("Saskatoon", 52.13, -106.67),
    ("Regina", 50.45, -104.61),
    ("Boise", 43.62, -116.20),
    ("Spokane", 47.66, -117.43),
    ("Billings", 45.78, -108.50),
    ("Fargo", 46.88, -96.79),
    ("Milwaukee", 43.04, -87.91),
    ("Cleveland", 41.50, -81.69),
]

print(len(CITIES), "cities")


## 5. Pull winter histories (Open-Meteo)

Daily → winter aggregates. **Winter year Y** = Dec (Y-1) + Jan Y + Feb Y.


In [ ]:
import time
import requests

START = "1960-01-01"
END = "2025-12-31"
ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"


def fetch_city_daily(lat, lon, retries=3):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START,
        "end_date": END,
        "daily": "temperature_2m_mean,precipitation_sum,snowfall_sum",
        "timezone": "auto",
    }
    for attempt in range(retries):
        r = requests.get(ARCHIVE, params=params, timeout=120)
        if r.status_code == 200:
            return r.json()
        time.sleep(2 * (attempt + 1))
    r.raise_for_status()


def daily_to_winters(payload):
    daily = payload["daily"]
    df = pd.DataFrame(
        {
            "date": pd.to_datetime(daily["time"]),
            "tmean": daily["temperature_2m_mean"],
            "precip": daily["precipitation_sum"],
            "snow": daily["snowfall_sum"],
        }
    )
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    # Dec belongs to next year's winter
    df["winter_year"] = np.where(df["month"] == 12, df["year"] + 1, df["year"])
    winter = df[df["month"].isin([12, 1, 2])].copy()

    g = winter.groupby("winter_year").agg(
        tmean=("tmean", "mean"),
        precip=("precip", "sum"),
        snow=("snow", "sum"),
        n_days=("tmean", "count"),
    )
    # roughly full DJF
    g = g[g["n_days"] >= 85]
    return g


# DJF ONI: middle month = January of winter_year
oni_jan = (
    oni_df.assign(time=pd.to_datetime(oni_df["time"]))
    .assign(month=lambda d: d["time"].dt.month, year=lambda d: d["time"].dt.year)
)
oni_by_winter = (
    oni_jan[oni_jan["month"] == 1]
    .set_index("year")["oni"]
    .astype(float)
)
print("ONI winters:", oni_by_winter.index.min(), "->", oni_by_winter.index.max())


In [ ]:
city_winters = {}  # name -> DataFrame indexed by winter_year

for i, (name, lat, lon) in enumerate(CITIES, 1):
    print(f"[{i}/{len(CITIES)}] {name} ...", end=" ")
    try:
        payload = fetch_city_daily(lat, lon)
        winters = daily_to_winters(payload)
        city_winters[name] = winters
        print(f"{len(winters)} winters")
    except Exception as e:
        print("FAIL", e)
    time.sleep(0.4)  # be polite to the free API

print("loaded", len(city_winters), "/", len(CITIES))


## 6. ONI → impacts per city

For each variable: anomaly = winter value − city long-term winter mean.  
Fit a line vs DJF ONI. Plug in `forecast_oni`.

**Confidence:** among past El Niño winters (DJF ONI ≥ 0.5), fraction whose anomaly **sign** matched the forecast anomaly sign.


In [ ]:
def phrase(var, anom):
    if var == "temp":
        if anom > 0.3:
            return "warmer than usual"
        if anom < -0.3:
            return "colder than usual"
        return "near-normal temperatures"
    if var == "precip":
        if anom > 10:
            return "wetter than usual"
        if anom < -10:
            return "drier than usual"
        return "near-normal precipitation"
    # snow (cm)
    if anom > 5:
        return "snowier than usual"
    if anom < -5:
        return "less snow than usual"
    return "near-normal snowfall"


def confidence_tag(score):
    if score >= 0.7:
        return "strong"
    if score >= 0.55:
        return "moderate"
    return "mixed"


def fit_city(name, lat, lon, winters, forecast_oni, oni_by_winter):
    # align winters with ONI
    common = winters.join(oni_by_winter.rename("oni"), how="inner")
    common = common.dropna()
    if len(common) < 20:
        return None

    out = {
        "city": name,
        "lat": lat,
        "lon": lon,
        "n_winters": int(len(common)),
        "forecast_oni": float(forecast_oni),
        "winter": WINTER_LABEL,
    }

    confidences = []
    for var, key, unit_scale in [
        ("temp", "tmean", 1.0),
        ("precip", "precip", 1.0),
        ("snow", "snow", 1.0),
    ]:
        clim = float(common[key].mean())
        anom = common[key] - clim
        # linear: anom ≈ a + b * oni
        b, a = np.polyfit(common["oni"].to_numpy(), anom.to_numpy(), 1)
        pred = float(a + b * forecast_oni)

        # El Niño winters direction agreement
        enso = common[common["oni"] >= 0.5]
        if len(enso) >= 5 and abs(pred) > 1e-6:
            signs_ok = np.sign(enso[key] - clim) == np.sign(pred)
            # if pred near 0, mixed
            conf = float(signs_ok.mean())
        else:
            conf = 0.5
        confidences.append(conf)

        out[f"{var}_anom"] = pred
        out[f"{var}_phrase"] = phrase(var, pred)
        out[f"{var}_slope"] = float(b)

    out["confidence"] = float(np.mean(confidences))
    out["confidence_tag"] = confidence_tag(out["confidence"])
    return out


impacts = []
coords = {n: (la, lo) for n, la, lo in CITIES}
for name, winters in city_winters.items():
    lat, lon = coords[name]
    row = fit_city(name, lat, lon, winters, forecast_oni, oni_by_winter)
    if row:
        impacts.append(row)
    else:
        print("skip (too few winters):", name)

impacts_df = pd.DataFrame(impacts)
print(impacts_df[["city", "temp_anom", "precip_anom", "snow_anom", "confidence_tag"]].head(10))
print("cities in impacts:", len(impacts_df))


## 7. Write `impacts.json`


In [ ]:
import json

payload = {
    "winter": WINTER_LABEL,
    "forecast_oni": float(forecast_oni),
    "model_checkpoint": str(ckpt_path.name),
    "model_lead": int(ckpt.get("lead", LEAD)),
    "input_end": str(last_month.date()),
    "disclaimer": (
        "Model forecast fed through a historical ONI–city relationship. "
        "Not an official outlook; mid-latitude signal can be weak."
    ),
    "cities": impacts,
}

out_json = OUT_DIR / "impacts.json"
out_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")
# also next to data for the web app later
(DATA_DIR / "impacts.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Wrote", out_json)
print("Wrote", DATA_DIR / "impacts.json")
print(f"winter {WINTER_LABEL}  ONI forecast={forecast_oni:.3f}  cities={len(impacts)}")
print("Day 4 checkpoint: impacts table ready.")


## 8. Quick map sanity check

Dots colored by temperature anomaly (blue = colder, red = warmer).


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(
    impacts_df["lon"],
    impacts_df["lat"],
    c=impacts_df["temp_anom"],
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    s=60,
    edgecolors="k",
    linewidths=0.3,
)
plt.colorbar(sc, ax=ax, label="temp anomaly (°C)")
ax.set_xlabel("lon")
ax.set_ylabel("lat")
ax.set_title(f"Winter {WINTER_LABEL} temp anomaly from ONI={forecast_oni:.2f}")
ax.set_xlim(-170, -50)
ax.set_ylim(15, 65)
plt.tight_layout()
fig.savefig(OUT_DIR / "impacts_temp_preview.png", dpi=140)
plt.show()
print("Wrote", OUT_DIR / "impacts_temp_preview.png")
